# 14.6 輸入輸出權重應該共享嗎？

# 第 14 章：逐項理解現代文字模型的設計

前置：第4章基準。這章像替熟悉的車逐一換零件，每節可獨立進入；不要同時換六項再猜哪項有效。結構／數值可先CPU驗證，品質與成本需匹配預算後訓練。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：成對feature像鐘面指針：用相對旋轉記位置**

單一pair的旋轉示意；不同pair有不同角速度。

這張圖保留靜態標註；先核對箭頭、位置與每個數字的意思。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rope.svg")))

**先想一想：**兩個weight初始相等，訓練時就會保持相等嗎？

輸入查字典與輸出給候選打分可共用同一張表，稱weight tying。它省參數但限制兩種任務的自由度，是否更好需實測。

**把直覺寫成數學：**E:[V,D]；輸出用xEᵀ。共享是同一Parameter，不是複製初值。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig

for tied in (False, True):
    model = TinyLM(ModelConfig(width=8, tied=tied))
    print(tied, model.description()["parameters"], model.output.weight is model.embedding.weight)
    assert (model.output.weight is model.embedding.weight) == tied

**如何讀結果：**untied但複製值仍是兩個可獨立更新的參數；量化替換後共享需再說明。

**只改一件事：**只增加詞表V，觀察省下多少參數。
